# Test: seed dimensions

Runs `src/setup` and `src/load_silver_dims`, then checks that each dimension table holds exactly the rows of its
seed CSV (`dim_market`, `dim_commodity`, `dim_date`): `EXCEPT ALL` in both directions returns nothing.
`updated_at` is not compared; the CSV does not have it.

Tests are plain pytest functions run inside this notebook with `ipytest`.
The notebook fails if any test fails, so it can be used as a job task.

Cleanup: after the tests, the tables and schemas that this run created are dropped.
Existing ones (and their data) are left as they were.

In [ ]:
%pip install pytest ipytest

## State before setup

Recorded before `src/setup` runs, so cleanup only drops what this run creates.

In [ ]:
test_catalog = spark.catalog.currentCatalog()

# Tables and schemas that setup creates, tables first so they are dropped before their schema.
SETUP_TABLES = [
    f"{test_catalog}.silver.dim_market",
    f"{test_catalog}.silver.dim_commodity",
    f"{test_catalog}.silver.dim_date",
    f"{test_catalog}.bronze.raw_mandi_transactions",
]
SETUP_SCHEMAS = [f"{test_catalog}.silver", f"{test_catalog}.bronze"]

tables_existed = {name: spark.catalog.tableExists(name) for name in SETUP_TABLES}
schemas_existed = {
    name: len(spark.sql(f"SHOW SCHEMAS IN {test_catalog} LIKE '{name.split('.')[1]}'").collect()) == 1
    for name in SETUP_SCHEMAS
}

## Run setup and seed

In [ ]:
%run ../src/setup

In [ ]:
%run ../src/load_silver_dims

## pytest setup

In [ ]:
import sys

import ipytest
import pytest

# Keep pytest from writing __pycache__ into the Git folder.
sys.dont_write_bytecode = True

ipytest.autoconfig(raise_on_error=True, addopts=["-p", "no:cacheprovider"])

In [ ]:
@pytest.fixture(scope="module", autouse=True)
def drop_created_objects() -> "Generator[None, None, None]":
    """After the tests, drop the tables and schemas, only those this run created.

    No CASCADE: dropping a schema fails instead of removing anything else in it.
    """
    yield

    for name in SETUP_TABLES:
        if not tables_existed[name]:
            spark.sql(f"DROP TABLE IF EXISTS {name}")
    for name in SETUP_SCHEMAS:
        if not schemas_existed[name]:
            spark.sql(f"DROP SCHEMA IF EXISTS {name}")

## Test: table matches file

`seeds` and `seed_dir` come from `src/load_silver_dims`; each test gets the full file path and the table name. Both sides are compared as strings:
the CSV as read, the table with its columns cast to STRING (this also checks the casts).

In [ ]:
# (file path, table name) for each seed.
SEED_FILES = [(f"{seed_dir}/{file_name}", table_name) for file_name, table_name, _ in seeds]


def read__seed_file(file_path: str) -> "DataFrame":
    """Read a seed CSV with all columns as STRING."""
    return (
        spark.read
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"file:{file_path}" if file_path.startswith("/Workspace") else file_path)
    )


@pytest.mark.parametrize("file_path, table_name", SEED_FILES)
def test__table_matches_seed_file(file_path: str, table_name: str) -> None:
    """The table has exactly the rows of the seed file (EXCEPT ALL, both ways)."""
    file_df = read__seed_file(file_path)
    table_df = spark.table(table_name).drop("updated_at")
    table_df = table_df.select(*[table_df[c].cast("string").alias(c) for c in table_df.columns])

    assert file_df.count() > 0
    assert file_df.exceptAll(table_df).count() == 0
    assert table_df.exceptAll(file_df).count() == 0

## Run

In [ ]:
ipytest.run("-v")